# XGBoost Volatility Forecasting Model Development

## Objective
Fine-tune the XGBoost model to minimize RMSE on forward realized volatility forecasts.

## Constraints
- **Training data**: Only use data before 2020-01-01 (backtest start)
- **Validation**: Time-series cross-validation (no future data leakage)
- **Target**: Forward 30-day realized volatility

## Outline
1. Data Loading & Preparation
2. Feature Engineering Exploration
3. Baseline Model Performance
4. Hyperparameter Tuning
5. Best Model Evaluation
6. Bias Analysis
7. Export Best Parameters


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ML imports
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr

# For GARCH comparison
from arch import arch_model

print("Imports complete")

HORIZONS = [1, 2, 5, 10, 22, 30, 60]
LOOKBACK = 30

## 1. Data Loading & Preparation


In [ ]:
# Load price and VIX data
prices_df = pd.read_parquet('../data/processed/spy_prices.parquet')
prices_df['date'] = pd.to_datetime(prices_df['date'])
prices_df = prices_df.sort_values('date').reset_index(drop=True)

vix_df = pd.read_parquet('../data/processed/vix_data.parquet')
vix_df['date'] = pd.to_datetime(vix_df['date'])
vix_df = vix_df.sort_values('date').reset_index(drop=True)

# Calculate log returns (in percentage terms to match GARCH convention)
prices_df['log_ret'] = np.log(prices_df['close'] / prices_df['close'].shift(1)) * 100

# Initial VIX filter
vix_df = vix_df[vix_df['date'] > pd.Timestamp('1993-01-28')]

# Define cutoff date - NO DATA AFTER THIS FOR TRAINING
TEST_START = pd.Timestamp('2015-01-01')
BACKTEST_START = pd.Timestamp('2020-01-01')

# Split data
no_backtest_df = prices_df[prices_df['date'] < BACKTEST_START].copy()
no_backtest_vix_df = vix_df[vix_df['date'] < BACKTEST_START].copy()

train_df = no_backtest_df[no_backtest_df['date'] < TEST_START].copy()
test_df = no_backtest_df[no_backtest_df['date'] >= TEST_START].copy()
val_df = prices_df[prices_df['date'] >= BACKTEST_START].copy()

vix_train_df = vix_df[vix_df['date'] < TEST_START].copy()
vix_test_df = vix_df[vix_df['date'] >= TEST_START].copy()
vix_val_df = vix_df[vix_df['date'] >= BACKTEST_START].copy()


print(f"Training data: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df):,} days)")
print(f"Testing data: {test_df['date'].min()} to {test_df['date'].max()} ({len(test_df):,} days)")
print(f"Back-Test data: {val_df['date'].min()} to {val_df['date'].max()} ({len(val_df):,} days)")
print(f"Training data (VIX): {vix_train_df['date'].min()} to {vix_train_df['date'].max()} ({len(vix_train_df):,} days)")
print(f"Testing data (VIX): {vix_test_df['date'].min()} to {vix_test_df['date'].max()} ({len(vix_test_df):,} days)")
print(f"Back-Test data (VIX): {vix_val_df['date'].min()} to {vix_val_df['date'].max()} ({len(vix_val_df):,} days)")

In [ ]:
def create_features(returns: pd.Series, vix: pd.Series, lookback, horizon) -> pd.DataFrame:
    returns = returns.reset_index(drop=True)
    vix = vix.reset_index(drop=True)


    # ===========================================
    # VIX Features
    # ===========================================
    yesterday_vix = vix.shift(1)

    # ===========================================
    # REALIZED VOLATILITY FEATURES
    # ===========================================
    rv_1d = (np.abs(returns) * np.sqrt(252)).shift(1)            # Yesterday's |return|
    rv_5d = (returns.rolling(5).std() * np.sqrt(252)).shift(1)   # Past 5 days (t-5 to t-1)
    rv_22d = (returns.rolling(22).std() * np.sqrt(252)).shift(1) # Past 22 days (t-22 to t-1)
    rv_60d = (returns.rolling(60).std() * np.sqrt(252)).shift(1) # Past 60 days (t-60 to t-1)
    rv_120d = (returns.rolling(120).std() * np.sqrt(252)).shift(1) # Past 120 days (t-120 to t-1)

    rv_1_5_mom = rv_1d - rv_5d
    rv_1_5_rate = rv_1d / rv_5d

    rv_5_22_mom = rv_5d - rv_22d
    rv_5_22_rate = rv_5d / rv_22d

    # ===========================================
    # LEVERAGE EFFECT (Black 1976)
    # ===========================================
    ret_1d = returns.shift(1)  # Yesterday's return
    ret_5d = returns.rolling(5).sum().shift(1)   # Cumulative return t-5 to t-1
    ret_22d = returns.rolling(22).sum().shift(1) # Cumulative return t-22 to t-1
    ret_60d = returns.rolling(60).sum().shift(1) # Cumulative return t-60 to t-1
    ret_120d = returns.rolling(120).sum().shift(1) # Cumulative return t-120 to t-1
    
    # ===========================================
    # ASYMMETRIC/SIGNED VOLATILITY (Patton & Sheppard 2015)
    # ===========================================
    neg_returns = returns.clip(upper=0)  # Negative returns only
    pos_returns = returns.clip(lower=0)  # Positive returns only
    
    # Realized semivariance: sqrt(sum(r^2)) * sqrt(252) / 100
    rsv_neg_5d = (np.sqrt((neg_returns**2).rolling(5).sum()) * np.sqrt(252)).shift(1)
    rsv_pos_5d = (np.sqrt((pos_returns**2).rolling(5).sum()) * np.sqrt(252)).shift(1)
    rsv_ratio = rsv_neg_5d / (rsv_pos_5d + 1e-8)  # Asymmetry ratio

    max_abs_ret_5d = np.abs(returns).rolling(5).max().shift(1)

    # ===========================================
    # BUILD DATAFRAME
    # ===========================================
    df = pd.DataFrame({
        'yesterday_vix': yesterday_vix,
        'rv_1d': rv_1d,
        'rv_5d': rv_5d,
        'rv_22d': rv_22d,        
        'rv_60d': rv_60d,
        'rv_120d': rv_120d,
        'rv_1_5_mom': rv_1_5_mom,
        'rv_1_5_rate': rv_1_5_rate,
        'rv_5_22_mom': rv_5_22_mom,
        'rv_5_22_rate': rv_5_22_rate,
        'ret_1d': ret_1d,
        'ret_5d': ret_5d,
        'ret_22d': ret_22d,
        'ret_60d': ret_60d,
        'ret_120d': ret_120d,
        'rsv_neg_5d': rsv_neg_5d,
        'rsv_pos_5d': rsv_pos_5d,
        'rsv_ratio': rsv_ratio, 
        'max_abs_ret_5d': max_abs_ret_5d,
        'horizon': horizon
    })
    
    # Trim to start after lookback (ensures enough history for all features)
    df = df.iloc[lookback:].reset_index(drop=True)
    
    return df


def create_targets(returns: pd.Series, lookback, horizon) -> np.ndarray:
    rolling_vol = returns.rolling(window=horizon).std() * np.sqrt(252)
    targets = rolling_vol.shift(-horizon)
    target_slice = targets.iloc[lookback:-horizon]
    
    return target_slice.values


def create_feature_targets(returns, vix, horizons, lookback):
    # 1. Create Features and Targets for each horizon
    all_X, all_y = [], []
    
    for h in horizons:
        X = create_features(returns, vix, lookback=lookback, horizon=h)
        y = create_targets(returns, lookback=lookback, horizon=h)
        
        # Align X and y: targets are shorter because they need forward data
        # X has (n - lookback) rows, y has (n - lookback - horizon) rows
        # Trim X from the end to match y
        n_samples = len(y)
        if n_samples > 0:
            X = X.iloc[:n_samples]
            all_X.append(X)
            all_y.append(y)
    
    if len(all_X) == 0:
        raise ValueError("Not enough data to create training samples")
    
    # 2. Combine all horizons
    X_combined = pd.concat(all_X, ignore_index=True)
    y_combined = np.concatenate(all_y)
    
    # 3. Remove rows with NaN values
    mask = ~(X_combined.isna().any(axis=1) | np.isnan(y_combined))
    X_combined = X_combined[mask]
    y_combined = y_combined[mask]

    return X_combined, y_combined


def fit(returns: pd.Series, vix: pd.Series, min_train_size: int = 30, horizons: list = [1, 2, 5, 10, 22, 30, 60, 120], lookback: int = 60) -> tuple:

    if len(returns) < min_train_size:
        raise ValueError(f"Need at least {min_train_size} returns to fit ML model")
    
    # Randomized Search for CV
    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=115,
        objective='reg:squarederror'
    )
    # {'subsample': 0.9, 'n_estimators': 700, 'max_depth': 7, 'learning_rate': 0.2, 'colsample_bytree': 0.9}
    param_grid = {
        'n_estimators': [300],
        'max_depth': [3],
        'learning_rate': [0.05],
        'subsample': [0.8, 0.9],
        'colsample_bytree': [0.8, 0.9]
    }
    
    # Time-series cross-validation
    tscv = TimeSeriesSplit(n_splits=2)  
    
    regressor = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=100,  # Number of parameter settings to sample
        cv=tscv,
        scoring='neg_root_mean_squared_error',
        verbose=2,
        random_state=115,
        n_jobs=-1
    )

    # Create features and targets for train data
    X_combined, y_combined = create_feature_targets(returns, vix, horizons, lookback)

    # Fit the model
    regressor.fit(X_combined, y_combined)
    
    # Save feature names
    feature_names = X_combined.columns.tolist()
    
    print(f"Model trained on {len(X_combined):,} samples across {len(horizons)} horizons")
    print(f"Features: {feature_names}")
    
    return regressor, feature_names


# ===========================================
# TEST THE FUNCTIONS
# ===========================================
print("Testing fit() function...")
returns = train_df['log_ret'].dropna()
vix_train = vix_train_df['vix_close'].dropna()
vix_test = vix_test_df['vix_close'].dropna()

# Fit with default model
fitted_model, feature_cols = fit(returns, vix_train)

print(f"\nModel fitted successfully!")
print(f"Feature columns: {feature_cols}")
print(f"Best Parameters found: {fitted_model.best_params_}")

In [ ]:
# Store best parameters
BEST_PARAMS = fitted_model.best_params_.copy()
BEST_PARAMS['objective'] = 'reg:squarederror'
BEST_PARAMS['random_state'] = 115

# Create train and test features
X_train, y_train = create_feature_targets(train_df['log_ret'].dropna(), vix_train, HORIZONS, LOOKBACK)
X_test, y_test = create_feature_targets(test_df['log_ret'].dropna(), vix_test, HORIZONS, LOOKBACK)

# Train best model on full training data
best_model = xgb.XGBRegressor(**BEST_PARAMS)
best_model.fit(X_train, y_train)

# Predictions on training and test
train_preds = best_model.predict(X_train)
test_preds = best_model.predict(X_test)

In [ ]:
def evaluate_by_horizon(returns, model, horizons, lookback):
    """
    Evaluate model performance separately for each horizon.
    
    Returns:
        DataFrame with metrics by horizon
    """
    results = []
    
    for h in horizons:
        # Create features/targets for this horizon only
        X = create_features(returns, vix_test, lookback=lookback, horizon=h)
        y = create_targets(returns, lookback=lookback, horizon=h)
        
        # Align
        n_samples = len(y)
        X = X.iloc[:n_samples]
        
        # Remove NaN
        mask = ~(X.isna().any(axis=1) | np.isnan(y))
        X = X[mask]
        y = y[mask]
        
        if len(X) == 0:
            continue
        
        # Predict
        preds = model.predict(X)
        
        # Calculate metrics
        metrics = evaluate_forecast(y, preds, f"Horizon={h}d", verbose=False)
        metrics['horizon'] = h
        metrics['n_samples'] = len(y)
        results.append(metrics)
    
    return pd.DataFrame(results)

def evaluate_forecast(y_true, y_pred, name="Model", verbose=True):
    """Calculate and display forecast metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = (y_pred - y_true).mean()
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    spearman = spearmanr(y_true, y_pred)[0]
    
    if verbose:
        print(f"\n{name}:")
        print(f"  RMSE:        {rmse:.6f}")
        print(f"  MAE:         {mae:.6f}")
        print(f"  Bias:        {bias:+.6f}")
        print(f"  Correlation: {corr:.4f}")
        print(f"  Spearman:    {spearman:.4f}")
    
    return {'name': name, 'rmse': rmse, 'mae': mae, 'bias': bias, 'corr': corr, 'spearman': spearman}

# ===========================================
# PERFORMANCE BY HORIZON
# ===========================================
print("\n" + "="*60)
print("PERFORMANCE BY HORIZON (Test Set)")
print("="*60)

test_returns = test_df['log_ret'].dropna()
horizon_results = evaluate_by_horizon(test_returns, best_model, HORIZONS, LOOKBACK)

# Display as formatted table
display_cols = ['horizon', 'n_samples', 'rmse', 'mae', 'bias', 'corr', 'spearman']
horizon_display = horizon_results[display_cols].copy()
horizon_display.columns = ['Horizon (days)', 'Samples', 'RMSE', 'MAE', 'Bias', 'Correlation', 'Spearman']
horizon_display = horizon_display.set_index('Horizon (days)')
display(horizon_display.round(2))

# Visualize performance by horizon
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# RMSE by horizon
ax = axes[0]
ax.bar(horizon_results['horizon'].astype(str), horizon_results['rmse'], color='steelblue', edgecolor='black')
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('RMSE')
ax.set_title('RMSE by Horizon')
ax.legend()

# Correlation by horizon
ax = axes[1]
ax.bar(horizon_results['horizon'].astype(str), horizon_results['corr'], color='green', edgecolor='black')
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('Correlation')
ax.set_title('Correlation by Horizon')
ax.legend()

# Bias by horizon
ax = axes[2]
colors = ['green' if b < 0 else 'red' for b in horizon_results['bias']]
ax.bar(horizon_results['horizon'].astype(str), horizon_results['bias'], color=colors, edgecolor='black')
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('Bias')
ax.set_title('Bias by Horizon\n(Green=Underestimate, Red=Overestimate)')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig('../results/xgb_performance_by_horizon.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(15).plot(kind='barh', x='feature', y='importance', ax=ax, legend=False)
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances (Tuned XGBoost)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSorted Features:")
display(importance)


In [ ]:
# ===========================================
# SINGLE HORIZON VISUALIZATION
# Change PLOT_HORIZON to explore different horizons
# ===========================================
PLOT_HORIZON = 30  # <-- CHANGE THIS: Options from HORIZONS list (1, 5, 30, 60, 120)

# Create features/targets for selected horizon
X_test_single = create_features(test_df['log_ret'].dropna(), vix_test, lookback=LOOKBACK, horizon=PLOT_HORIZON)
y_test_single = create_targets(test_df['log_ret'].dropna(), lookback=LOOKBACK, horizon=PLOT_HORIZON)

# Align X and y
n_samples = len(y_test_single)
X_test_single = X_test_single.iloc[:n_samples]

# Remove NaN
mask = ~(X_test_single.isna().any(axis=1) | np.isnan(y_test_single))
X_test_single = X_test_single[mask]
y_test_single = y_test_single[mask]

# Predict
test_preds_single = best_model.predict(X_test_single)

# Get corresponding dates
test_dates = test_df['date'].iloc[LOOKBACK:LOOKBACK + len(y_test_single)].values

# Calculate metrics for this horizon
single_results = evaluate_forecast(y_test_single, test_preds_single, f"Horizon={PLOT_HORIZON}d", verbose=False)

# ===========================================
# PLOTS
# ===========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax = axes[0]
ax.scatter(y_test_single, test_preds_single, alpha=0.3, s=10)
ax.plot([y_test_single.min(), y_test_single.max()], 
        [y_test_single.min(), y_test_single.max()], 'r--', lw=2, label='Perfect')
ax.set_xlabel('Actual Forward RV')
ax.set_ylabel('Predicted RV')
ax.set_title(f'Predicted vs Actual (Horizon={PLOT_HORIZON}d)\nCorr: {single_results["corr"]:.3f}')
ax.legend()

# Time series
ax = axes[1]
ax.plot(test_dates, y_test_single, label='Actual RV', alpha=0.7)
ax.plot(test_dates, test_preds_single, label='Predicted RV', alpha=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Volatility (Annualized)')
ax.set_title(f'Time Series Comparison (Horizon={PLOT_HORIZON}d)')
ax.legend()
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'../results/xgb_evaluation_h{PLOT_HORIZON}.png', dpi=150, bbox_inches='tight')
plt.show()

# Print metrics
print(f"\n{'='*50}")
print(f"HORIZON = {PLOT_HORIZON} DAYS")
print(f"{'='*50}")
print(f"  Samples:     {len(y_test_single):,}")
print(f"  RMSE:        {single_results['rmse']:.4f}")
print(f"  MAE:         {single_results['mae']:.4f}")
print(f"  Bias:        {single_results['bias']:+.4f}")
print(f"  Correlation: {single_results['corr']:.4f}")
print(f"  Spearman:    {single_results['spearman']:.4f}")


## 4. Bias Analysis


In [ ]:
# Bias Analysis using single-horizon predictions (from previous cell)
# We use y_test_single and test_preds_single which are aligned with dates

# Create analysis DataFrame
test_analysis = pd.DataFrame({
    'date': test_dates,
    'actual_rv': y_test_single,
    'pred_rv': test_preds_single,
    'error': test_preds_single - y_test_single
})

# Classify by actual volatility level (values are in percentage scale from std * sqrt(252))
# Convert to decimal for regime classification
test_analysis['actual_rv_decimal'] = test_analysis['actual_rv'] / 100

test_analysis['vol_regime'] = pd.cut(
    test_analysis['actual_rv_decimal'],
    bins=[0, 0.10, 0.15, 0.25, 1.0],
    labels=['Very Low (<10%)', 'Low (10-15%)', 'Normal (15-25%)', 'Stress (>25%)']
)

# Bias by regime
regime_bias = test_analysis.groupby('vol_regime').agg({
    'error': ['mean', 'std', 'count'],
    'actual_rv': 'mean'
}).round(4)

print("Forecast Bias by Volatility Regime:")
display(regime_bias)

# Visualize bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution
ax = axes[0]
test_analysis['error'].hist(bins=50, ax=ax, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=2)
mean_error = test_analysis['error'].mean()
ax.axvline(mean_error, color='green', linestyle='--', linewidth=2, 
           label=f'Mean Bias: {mean_error:+.2f}%')
ax.set_xlabel('Forecast Error (Pred - Actual)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of Forecast Errors (Horizon={PLOT_HORIZON}d)')
ax.legend()

# Error over time
ax = axes[1]
test_analysis['date'] = pd.to_datetime(test_analysis['date'])
rolling_error = test_analysis.set_index('date')['error'].rolling('30D').mean()
ax.plot(rolling_error.index, rolling_error.values, label='30-day rolling mean error')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.fill_between(rolling_error.index, rolling_error.values, 0, alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Forecast Error')
ax.set_title('Forecast Bias Over Time')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../results/xgb_bias_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print(f"\nBias Summary (Horizon={PLOT_HORIZON}d):")
print(f"  Mean Bias:  {mean_error:+.4f}")
print(f"  Std Error:  {test_analysis['error'].std():.4f}")
print(f"  % Positive: {(test_analysis['error'] > 0).mean()*100:.1f}%")